In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, hashlib, subprocess, time
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','features_ugr16']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, features_ugr16 as fu
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - rebuild the July source and August target EXACTLY as nb15 did, so the
# saved partition indices align, then encode features with the source-learned
# vocab. Multiclass here: background + dos + scan11 + scan44 + nerisbotnet.
# =============================================================================
UGR = config.DATASETS_DIR / 'ugr16'
src = pd.read_parquet(UGR/'july_week5.parquet')
tgt = pd.read_parquet(UGR/'august_week1.parquet')
for d in (src, tgt): d['label'] = d['label'].astype(str).str.strip().str.lower()
KEEP = ['background','dos','scan11','scan44','nerisbotnet']
src = src[src.label.isin(KEEP)].reset_index(drop=True)
tgt = tgt[tgt.label.isin(KEEP)].reset_index(drop=True)

CLASSES = sorted(KEEP)                      # fixed column order for probabilities
C2I = {c:i for i,c in enumerate(CLASSES)}

# reproduce the nb15 stratified split (same seed) to recover the partitions
PARTITION_SEED_UGR = 20260725
def stratified_split(df, fractions, seed, col='label'):
    rng=np.random.default_rng(seed); names=list(fractions)
    fr=np.array([fractions[k] for k in names],float); big=names[int(np.argmax(fr))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(fr*n).astype(int); c[names.index(big)]+=n-c.sum(); k=0
        for nm,q in zip(names,c): a.loc[idx[k:k+q]]=nm; k+=q
    return a
src = src.assign(partition=stratified_split(src, config.SPLIT_FRACTIONS, PARTITION_SEED_UGR).values)
parts = {k: src[src.partition==k] for k in config.SPLIT_FRACTIONS}

vocab = fu.build_vocab(parts['train'])
def Xy(df):
    X,_ = fu.encode(df, vocab)
    y = df['label'].map(C2I).to_numpy()
    return X, y

PROBS_DIR = config.DATA_DIR / 'ugr16_probs'; PROBS_DIR.mkdir(parents=True, exist_ok=True)
print('classes:', CLASSES)
print('source parts:', {k:len(v) for k,v in parts.items()}, '| target:', len(tgt))


classes: ['background', 'dos', 'nerisbotnet', 'scan11', 'scan44']
source parts: {'train': 240000, 'val': 40000, 'probcal': 60000, 'source_cal_pool': 60000} | target: 400000


In [3]:
# =============================================================================
# Cell 3 - fixed hyperparameters (logged deviation, same rationale as CIC) and
# the isotonic OvR calibrator (preregistration 6). Multiclass RF / XGB / MLP.
# =============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import f1_score
import xgboost as xgb

NCLS = len(CLASSES)
HP = {
  'rf':  dict(n_estimators=200, min_samples_leaf=2, n_jobs=-1),
  'xgb': dict(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method='hist',
              n_jobs=-1, eval_metric='mlogloss'),   # multiclass auto-detected from labels
  'mlp': dict(hidden_layer_sizes=(128,64), early_stopping=True, max_iter=60, batch_size=512),
}

def train_one(arch, Xtr, ytr, scal):
    if arch=='rf':  m=RandomForestClassifier(random_state=SEED, **HP['rf']); m.fit(Xtr,ytr)
    elif arch=='xgb': m=xgb.XGBClassifier(random_state=SEED, **HP['xgb']); m.fit(Xtr,ytr)
    else: m=MLPClassifier(random_state=SEED, **HP['mlp']); m.fit(scal.transform(Xtr),ytr)
    return m

def proba(m, arch, X, scal):
    P = m.predict_proba(scal.transform(X) if arch=='mlp' else X)
    # reindex columns to CLASSES order (integer labels 0..NCLS-1 == C2I order)
    full = np.zeros((len(X), NCLS));
    for j,cls in enumerate(m.classes_): full[:, int(cls)] = P[:, j]
    return full

def isotonic_ovr(prob_cal, y_cal, prob_apply):
    out=np.zeros_like(prob_apply)
    for j in range(NCLS):
        ir=IsotonicRegression(out_of_bounds='clip'); ir.fit(prob_cal[:,j], (y_cal==j).astype(float))
        out[:,j]=ir.predict(prob_apply[:,j])
    s=out.sum(1,keepdims=True); s[s==0]=1.0
    return out/s
print('model + calibration helpers ready; NCLS =', NCLS)



model + calibration helpers ready; NCLS = 5


In [4]:
# =============================================================================
# Cell 4 - RESUMABLE training: 3 architectures x 10 seeds = 30 models on the July
# source. Isotonic-calibrate on probcal, save calibrated probs on the source
# calibration pool and the full August target. Skips any already saved.
# =============================================================================
Dtr,Dpc,Dva = parts['train'],parts['probcal'],parts['val']
Sp = parts['source_cal_pool']

imp = SimpleImputer(strategy='median').fit(fu.encode(Dtr,vocab)[0])
Xtr = imp.transform(fu.encode(Dtr,vocab)[0]); ytr = Dtr['label'].map(C2I).to_numpy()
Xpc = imp.transform(fu.encode(Dpc,vocab)[0]); ypc = Dpc['label'].map(C2I).to_numpy()
Xva = imp.transform(fu.encode(Dva,vocab)[0]); yva = Dva['label'].map(C2I).to_numpy()
Xsp = imp.transform(fu.encode(Sp ,vocab)[0])
Xtg = imp.transform(fu.encode(tgt,vocab)[0])
scal = StandardScaler().fit(Xtr)

perf=[]; t0=time.time()
for SEED in config.SEEDS:
    for arch in ['rf','xgb','mlp']:
        out = PROBS_DIR/f'ugr16__{arch}__seed{SEED}.npz'
        if out.exists(): continue
        m = train_one(arch, Xtr, ytr, scal)
        f1 = f1_score(yva, proba(m,arch,Xva,scal).argmax(1), average='macro')
        pc = proba(m,arch,Xpc,scal)
        sp_cal = isotonic_ovr(pc, ypc, proba(m,arch,Xsp,scal))
        tg_cal = isotonic_ovr(pc, ypc, proba(m,arch,Xtg,scal))
        np.savez_compressed(out, srcpool=sp_cal, target=tg_cal, classes=np.array(CLASSES))
        perf.append({'dataset':'ugr16','seed':SEED,'arch':arch,'val_macro_f1':round(float(f1),4)})
        print(f'{arch:4s} seed {SEED:5d}  valF1={f1:.3f}  [{(time.time()-t0)/60:.1f} min]')
print(f'\ndone this run in {(time.time()-t0)/60:.1f} min; models trained now: {len(perf)}')


rf   seed    42  valF1=0.995  [0.9 min]
xgb  seed    42  valF1=0.971  [1.9 min]


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp  seed    42  valF1=0.826  [4.7 min]
rf   seed  1337  valF1=0.995  [5.5 min]
xgb  seed  1337  valF1=0.971  [6.6 min]


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp  seed  1337  valF1=0.834  [9.3 min]
rf   seed  2024  valF1=0.995  [10.2 min]
xgb  seed  2024  valF1=0.971  [11.2 min]
mlp  seed  2024  valF1=0.809  [12.7 min]
rf   seed     7  valF1=0.995  [13.6 min]
xgb  seed     7  valF1=0.971  [14.6 min]


...

mlp  seed  6021  valF1=0.835  [31.3 min]
rf   seed    88  valF1=0.995  [32.2 min]
xgb  seed    88  valF1=0.971  [33.2 min]


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp  seed    88  valF1=0.828  [35.9 min]
rf   seed  3407  valF1=0.995  [36.7 min]
xgb  seed  3407  valF1=0.971  [37.7 min]


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp  seed  3407  valF1=0.834  [40.5 min]
rf   seed 12345  valF1=0.995  [41.3 min]
xgb  seed 12345  valF1=0.971  [42.4 min]
mlp  seed 12345  valF1=0.829  [45.0 min]

done this run in 45.0 min; models trained now: 30


In [ ]:
# =============================================================================
# Cell 5 - record performance, log deviation, commit. Calibrated probs -> data/
# (gitignored); performance + manifest committed.
# =============================================================================
if perf:
    pf=pd.DataFrame(perf); pp=config.REPORTS_DIR/'model_performance_ugr16.csv'
    if pp.exists(): pf=pd.concat([pd.read_csv(pp),pf]).drop_duplicates(['seed','arch'],keep='last')
    pf.to_csv(pp,index=False); print(pf.groupby('arch')['val_macro_f1'].mean().round(3).to_string())

trained=sorted(p.name for p in PROBS_DIR.glob('*.npz'))
(config.REPORTS_DIR/'models_manifest_ugr16.json').write_text(json.dumps({
    'environment':'july_to_august','classes':CLASSES,'architectures':['rf','xgb','mlp'],
    'seeds':config.SEEDS,'hyperparameters_fixed':HP,
    'calibration':'one-vs-rest isotonic on probcal, renormalised (section 6)',
    'n_prob_files_present':len(trained),'n_expected_when_complete':3*len(config.SEEDS)}, indent=2))

dev=config.REPORTS_DIR/'deviations.md'
note=('\n## nb16 (UGR16) - fixed architecture-appropriate hyperparameters (RF/XGB/MLP), not a per-dataset '
      'macro-F1 grid (section 5), to bound compute; before any coverage.\n')
if dev.exists() and 'nb16 (UGR16)' not in dev.read_text():
    with open(dev,'a') as f: f.write(note)

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m',f'nb16: UGR16 multiclass model panel + isotonic calibration ({len(trained)}/{3*len(config.SEEDS)})')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
print(f'\nprob files: {len(trained)} / {3*len(config.SEEDS)}')


arch
mlp    0.830
rf     0.995
xgb    0.971
